# LAB 5: Conversational Search com RAG Pipeline no OpenSearch

## Objetivo da Aula

Construir um **pipeline de busca conversacional com memória de contexto** usando OpenSearch, demonstrando como integrar retrieval aumentado por geração (RAG) com capacidade de manter histórico de conversas multi-turno.

## Parte 1: Teoria Rápida

### Conversational Search

**Conversational Search** combina três componentes:

1. **Retrieval (BM25 + Vector)** - Busca híbrida recupera documentos relevantes
2. **RAG Processor** - Enriquece a pergunta com contexto dos documentos
3. **Memory Management** - Armazena histórico de mensagens com conversation_id

## Parte 2: Instalação de Dependências

In [ ]:
import subprocess
import sys

packages = [
    'opensearch-py>=2.2.0',
    'requests>=2.31.0',
    'openai>=1.0.0',
    'python-dotenv>=1.0.0',
    'pandas>=2.0.0',
    'langchain>=0.1.0',
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ Dependências instaladas!")

## Parte 3: Imports e Configuração

In [ ]:
import json
import os
from typing import Dict, List, Any, Optional
from datetime import datetime
import pandas as pd
from opensearchpy import OpenSearch, helpers
from opensearchpy.exceptions import OpenSearchException
import requests
import time
import warnings

warnings.filterwarnings('ignore')

def log(msg: str, level: str = "INFO"):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {level}: {msg}")

log("Imports realizados com sucesso")

## Parte 4: Conexão com OpenSearch

In [ ]:
OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER', 'admin')
OPENSEARCH_PASSWORD = os.getenv('OPENSEARCH_PASSWORD', 'admin')

VLLM_ENDPOINT = os.getenv('VLLM_ENDPOINT', 'http://localhost:8000/v1')
VLLM_MODEL = os.getenv('VLLM_MODEL', 'llama-3.1-8b-instruct')
VLLM_API_KEY = os.getenv('VLLM_API_KEY', 'not-needed')

try:
    client = OpenSearch(
        hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
        http_auth=(OPENSEARCH_USER, OPENSEARCH_PASSWORD),
        use_ssl=False,
        verify_certs=False,
        timeout=30
    )
    info = client.info()
    version = info.get('version', {}).get('number', 'unknown')
    log(f"Conectado ao OpenSearch v{version}")
except Exception as e:
    log(f"Erro ao conectar: {str(e)}", "ERROR")
    client = None

## Parte 5: Busca Híbrida

In [ ]:
def generate_simple_embedding(text: str, dim: int = 384):
    import hashlib
    hash_obj = hashlib.md5(text.encode())
    hash_hex = hash_obj.hexdigest()
    embedding = []
    for i in range(dim):
        byte_val = int(hash_hex[(i * 2) % len(hash_hex)] * 2, 16) if i < len(hash_hex) else 0
        embedding.append((byte_val % 100) / 100.0)
    return embedding

def hybrid_search(client, index_name: str, query_text: str, size: int = 5):
    query_embedding = generate_simple_embedding(query_text)
    search_body = {
        "size": size,
        "query": {
            "bool": {
                "should": [
                    {"multi_match": {"query": query_text, "fields": ["titulo^2", "conteudo"]}},
                    {"knn": {"conteudo_embedding": {"vector": query_embedding, "k": size}}}
                ]
            }
        }
    }
    try:
        response = client.search(index=index_name, body=search_body)
        return [{'id': hit['_id'], 'score': hit['_score'], 'source': hit['_source']} 
                for hit in response['hits']['hits']]
    except Exception as e:
        log(f"Erro na busca: {str(e)}", "ERROR")
        return []

## Parte 6: Pipeline Conversacional

In [ ]:
class ConversationalSearchPipeline:
    def __init__(self, client, index_name: str):
        self.client = client
        self.index_name = index_name
        self.conversation_history = []
        self.turn_count = 0
    
    def add_turn(self, user_query: str, assistant_response: str):
        self.turn_count += 1
        turn = {
            'turn': self.turn_count,
            'timestamp': datetime.now().isoformat(),
            'user_query': user_query,
            'assistant_response': assistant_response
        }
        self.conversation_history.append(turn)
    
    def search_and_augment(self, user_query: str):
        results = hybrid_search(self.client, self.index_name, user_query, size=5)
        return {
            'user_query': user_query,
            'search_results': results,
            'retrieved_sources': [r['source'] for r in results]
        }
    
    def get_history_dataframe(self):
        return pd.DataFrame(self.conversation_history) if self.conversation_history else pd.DataFrame()

if client:
    INDEX_NAME = "indice_juridico_conversacional"
    pipeline = ConversationalSearchPipeline(client, INDEX_NAME)
    log("Pipeline conversacional inicializado")
else:
    pipeline = None

## Parte 7: Turno 1 - Consulta Inicial

In [ ]:
if pipeline:
    turno1_query = "Quais são os direitos fundamentais garantidos?"
    print(f"TURNO 1: {turno1_query}")
    
    augmented = pipeline.search_and_augment(turno1_query)
    print(f"Documentos recuperados: {len(augmented['search_results'])}")
    
    resposta_turno1 = "Os direitos fundamentais incluem vida, liberdade, igualdade, segurança e propriedade. (CF/88, Art. 5º)"
    pipeline.add_turn(turno1_query, resposta_turno1)
    print(f"Assistant: {resposta_turno1}")
else:
    print("Pipeline não inicializado")

## Parte 8: Turno 2 - Follow-up

In [ ]:
if pipeline:
    turno2_query = "Como a privacidade é protegida?"
    print(f"TURNO 2: {turno2_query}")
    
    augmented = pipeline.search_and_augment(turno2_query)
    resposta_turno2 = "A privacidade é protegida pela LGPD e pela Constituição Federal. (LGPD + CF/88)"
    pipeline.add_turn(turno2_query, resposta_turno2)
    print(f"Assistant: {resposta_turno2}")
else:
    print("Pipeline não inicializado")

## Parte 9: Histórico da Conversa

In [ ]:
if pipeline and len(pipeline.conversation_history) > 0:
    df_history = pipeline.get_history_dataframe()
    print(f"Total de turnos: {len(df_history)}")
    print(f"\nHistórico:\n{df_history.to_string()}")
else:
    print("Histórico vazio ou pipeline não inicializado")

## Referências ABNT

BRASIL. Constituição Federal. 1988. Brasília, DF: Senado Federal.

BRASIL. Lei nº 12.527, de 18 de novembro de 2011. Lei de Acesso à Informação. Diário Oficial da União, Brasília, DF, 19 nov. 2011.

BRASIL. Lei nº 13.709, de 14 de agosto de 2018. Lei Geral de Proteção de Dados Pessoais (LGPD). Diário Oficial da União, Brasília, DF, 15 ago. 2018.

OPENEARCH PROJECT. OpenSearch Documentation. Disponível em: https://opensearch.org/docs/. Acesso em: 2026.